# Imports & Constants

In [1]:
%load_ext autoreload

In [2]:
%autoreload 2

In [3]:
import ast
import matplotlib.pyplot as plt
import numpy as np
import pickle
import pandas as pd
import random
import seaborn as sns
import statsmodels.api as sm
import contextlib
import joblib

In [4]:
from random import choices
from scipy import stats
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.cross_decomposition import PLSRegression
from tqdm.notebook import tqdm
from typing import Optional
from joblib import Parallel, delayed
from matplotlib.colors import ListedColormap

In [5]:
from jupyter_utils import style, mean_std, display_test, display_group_test, \
    scatter_annotate, show_corrtest_mask_corr, pointplot, pointplot_horizontal, add_grey, \
    prep_horizontal_pointplot_errobar_data, map_model, prep_LM_pointplot, draw_sample_with_replacement, t_test
from ortogonolize_utils import compute_coefficient

In [6]:
import warnings
warnings.filterwarnings(action='ignore', category=np.VisibleDeprecationWarning)
warnings.filterwarnings(action='ignore', message='All-NaN slice encountered')
warnings.filterwarnings(action='ignore', message='Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.')
warnings.filterwarnings(action='ignore', message='Mean of empty slice')
warnings.filterwarnings(action='ignore', category=stats.ConstantInputWarning)
warnings.filterwarnings(action='ignore', message='indexing past lexsort depth may impact performance')

In [7]:
sns.set_theme(style="whitegrid")

In [8]:
PATH = '/Users/galina.ryazanskaya/Downloads/thesis?/code?/processed_values'

In [9]:
PATH_FIG = '/Users/galina.ryazanskaya/Downloads/thesis?/figures/de/'

In [10]:
ERRORBAR = ('pi', 95)

In [11]:
DECIMAL = 7

# Load data

In [12]:
combined_data_averaged = pd.read_csv('/Users/galina.ryazanskaya/Downloads/thesis?/code?/processed_values/de_averaged.csv', index_col=0, header=[0, 1])

In [13]:
combined_data_averaged.columns = [c if '(' not in c else ast.literal_eval(c) for c in combined_data_averaged.columns]

In [14]:
combined_data_all = pd.read_csv('/Users/galina.ryazanskaya/Downloads/thesis?/code?/processed_values/de_all.csv', index_col=0, header=[0, 1, 2])

In [15]:
TASKS = ['anger', 'fear', 'happiness', 'sadness']

In [16]:
def task_data(df, task, keep_target=True, fill_synt=True):
    subset = df[task].dropna(axis=0, how='any')
    if task == 'fear' and 'KG_018' in subset.index:
        subset = subset.drop(['KG_018'])   # incorrect task
    if fill_synt:
        subset['syntactic'] = subset['syntactic'].fillna(0.0)
    if keep_target:
        subset = pd.concat([subset, df['target'].loc[subset.index]], axis=1)
    return subset

In [17]:
def apply_to_all_tasks(df, f, tasks=TASKS, to_df=True, *args, **kwargs):
    res = {}
    for task in tasks:
        data = task_data(df, task)
        res[task] = f(data, *args, **kwargs)
    if to_df:
        if all(isinstance(v, pd.Series) for v in res.values()):
            return pd.DataFrame(res)
        elif all(isinstance(v, pd.DataFrame) for v in res.values()):
            return pd.concat(list(res.values()), keys=list(res.keys()), names=['task'], axis=1)
        else:
            return res
    return res

In [18]:
group_means = combined_data_all.groupby([('target', 'target', 'group')]).mean().T
group_means.columns = ['control', 'NAP']
group_means.index.rename(['task', 'feature_group', 'feature'], inplace=True)
group_means.to_csv("feature_means_de.csv")

## For additional analysis

In [19]:
best_verbosity_metric = ('syntactic', 'mean_sent_len')

In [20]:
cols_to_test_additionally = [('lexical','LTR'), ('graph', 'LCC'), ('graph', 'number_of_edges'), ('syntactic', 'PART'), ('LM', 'glove_tf_lcoh'), ('LM', 'bert_pppl')]

In [21]:
default_covariates_r = [('target','sex_nr'), ('target','age'), ('target', 'Bildungsjahre'), ('target', 'IQ_kristallin')]

In [22]:
default_covariates_t = [('target','sex_nr'), ('target','age'), ('target', 'Bildungsjahre')]

In [23]:
def compute_OLS_rsquared(df, target_col, predictor_cols, round_decimal=DECIMAL):
#     df = df.loc[df[[target_col] + predictor_cols].dropna().index]
    X = df[predictor_cols]
    y = df[[target_col]]
    ols = LinearRegression()
#     pls2 = PLSRegression(n_components=min(2, len(predictor_cols)))
    ols.fit(X, y)
    r2 = ols.score(X, y)
    r_adjusted = 1 - (1 - r2) * (len(y) - 1) / (len(y) - X.shape[1] - 1)
    return np.round(r2, decimals=round_decimal), r_adjusted

In [24]:
def compute_OLS_rsquared_difference(df, target_col, predictor_cols, control_for_col, 
                                    default_covariates, r_control_only=None, round_decimal=DECIMAL):
    if r_control_only is None:
        r_control_only, _ = compute_OLS_rsquared(df, target_col, [control_for_col] + default_covariates, 
                                              round_decimal=round_decimal)
    r_pred_and_control, r_adj_pred_and_control = compute_OLS_rsquared(df, target_col, 
                                              [control_for_col] + default_covariates + predictor_cols, 
                                              round_decimal=round_decimal)
    r_pred_and_control = r_control_only if r_pred_and_control < -1000 else r_pred_and_control
    r_diff = r_pred_and_control - r_control_only
    r_diff = 0 if r_diff < 0 else r_diff
    
    # marginal model (single predictor, no covariates)
    r_marginal, _ = compute_OLS_rsquared(df, target_col, predictor_cols, round_decimal=round_decimal)
    
    return r_control_only, r_pred_and_control, r_diff, r_marginal, r_adj_pred_and_control

In [25]:
def compute_roc_auc(df_groups, target_col, predictor_column):
#     df_groups = df_groups.loc[df_groups[[target_col, predictor_column]].dropna().index]
    X = df_groups[[predictor_column]]
    y = df_groups[[target_col]]
    clf = LogisticRegression(solver="newton-cholesky", random_state=0, penalty=None)
    clf.fit(X, y)
    y_pred_prob = clf.predict_proba(X)
    return roc_auc_score(y, y_pred_prob[:, 1])

In [26]:
def compute_logistic_llh_null(df_groups, target_col, round_decimal=DECIMAL):
    y = df_groups[target_col].values
    classes = np.unique(y)  # sorted by numpy, same as sklearn convention
    n = len(y)
    counts = {c: (y == c).sum() for c in classes}
    llh_null = sum(
        count * np.log(count / n) 
        for count in counts.values() 
        if count > 0
    )
    return np.round(llh_null, decimals=round_decimal)

In [27]:
def compute_logistic_llh(df_groups, target_col, predictor_cols, round_decimal=DECIMAL):
    X = df_groups[predictor_cols]
    y = df_groups[[target_col]]
    clf = LogisticRegression(solver="newton-cholesky", random_state=0, penalty=None)
    clf.fit(X, y)
    y_pred_prob = clf.predict_proba(X)
    # accuracy 
    # clf.score(X, y)
    return np.round(-log_loss(y, y_pred_prob, normalize=False), decimals=round_decimal)

In [28]:
# McFadden's adjusted pseudo-R²
# https://www.statease.com/docs/v23.0/contents/advanced-topics/glm/adj-mcfadden-pseudo-r-squared/
def compute_ps_r_adj_llh(sample_llh_null, sample_llh_to_correct, k):
    return 1 - (sample_llh_to_correct - k) / sample_llh_null

In [29]:
def logistic_llh_difference(df_groups, target_col, predictor_cols, control_for_col, 
                            default_covariates, llh_control_only=None, llh_null=None,
                            round_decimal=DECIMAL):
    # control model (verbosity + default covariates)
    if llh_control_only is None:
        llh_control_only = compute_logistic_llh(
            df_groups, target_col, [control_for_col] + default_covariates, 
            round_decimal=round_decimal
        )
    
    # full model
    llh_pred_and_control = compute_logistic_llh(
        df_groups, target_col, 
        [control_for_col] + default_covariates + predictor_cols, 
        round_decimal=round_decimal
    )
    
    # LLH difference (full model vs control model)
    llh_diff = 2 * (llh_pred_and_control - llh_control_only) # likelihood ratio statistic
    llh_diff = 0 if llh_diff < 0 else llh_diff
    
    # intercept-only model
    if llh_null is None:
        llh_null = compute_logistic_llh_null(df_groups, target_col, round_decimal=round_decimal)
    # marginal model (single predictor, no covariates)
    llh_marginal = compute_logistic_llh(
        df_groups, target_col, predictor_cols, 
        round_decimal=round_decimal
    )
    llh_marginal_diff = 2 * (llh_marginal - llh_null)
    # NEW
    llh_marginal_diff = 0 if llh_marginal_diff < 0 else llh_marginal_diff

    ps_r_adj = compute_ps_r_adj_llh(llh_null, llh_pred_and_control, k=len([control_for_col] + default_covariates + predictor_cols))
    
    return llh_control_only, llh_pred_and_control, llh_diff, llh_marginal_diff, ps_r_adj

In [30]:
def drop_na_in_cols(df, cols):
    return df.loc[df[cols].dropna().index]

# Bootstrap

In [31]:
scale_cols = ['saps_total',
             'sans_total',
             'panss_pos',
             'panss_neg',
             'panss_o',
             'panss_total']

In [32]:
cols_LM = [col for col in combined_data_averaged.columns if col[0] == 'LM']
cols_synt = [col for col in combined_data_averaged.columns if col[0] == 'syntactic']
cols_lex = [col for col in combined_data_averaged.columns if col[0] == 'lexical']
cols_graph = [col for col in combined_data_averaged.columns if col[0] == 'graph']
cols_av = cols_LM + cols_synt + cols_lex + cols_graph

In [33]:
ORDERED_SCALES = ['panss_pos', 'panss_neg', 'panss_o', 'panss_total', 'saps_total', 'sans_total']

In [34]:
cols_to_correct_for = [('syntactic', 'mean_sent_len'), ('syntactic', 'n_sents'), ('lexical', 'n_words')]

In [35]:
# with open('processed_values/de_scales_samples_w_verbosity.pickle', 'wb') as f:
#     pickle.dump(reform_v, f)

In [36]:
with open('processed_values/de_scales_samples_w_verbosity.pickle', 'rb') as f:
    reform_v = pickle.load(f)

In [37]:
reformed_d_w_verbosity = pd.DataFrame(reform_v)

In [38]:
with open('processed_values/de_scales_samples_wo_o.pickle', 'rb') as f:
    reform = pickle.load(f)

In [39]:
reformed_d = pd.DataFrame(reform)

### Bootstrap for each task
- sample with replacement (keep group proportions)
    1. compute r for each scale for each iteration
    2. compute t test for each iteration
    3. compute multivariate linear and logistic regression
    4. compute delta r2 wrt best verbosity metric
    5. compute ROC-AUC

In [40]:
a = 0.05

In [41]:
def ttest(groups_df, target_col, cols_tasks=cols_av):
    s_t, res_t = display_group_test(groups_df, cols_tasks, target_col, test=stats.ttest_ind, stat_name='t',
                                    alpha=a)
    return res_t['t']

In [42]:
def try_bootstrap_iteration(sample: pd.DataFrame,
                            group: str,
                            cols_av: list[tuple[str, str]],
                            scale_cols: list[tuple[str, str]],
                            columns_to_correct_for: list[tuple[str, str]],
                            best_verbosity_metric: tuple[str, str],
                            default_covariates_r: list[tuple[str, str]],
                            default_covariates_t: list[tuple[str, str]],
                            cols_to_test_additionally: list[tuple[str, str]],
                            target_col_corr: list[tuple[str, str]]) -> dict[tuple[str, str, str], float]:
    # we need to put scale_independent into each scale outside this function to comly with the old format
    iteration_result: dict[tuple[str, str, str], float] = {}  # (target, scale, column) : metric
    group_target_col = ('target', group)
    t_test_res = ttest(sample, group_target_col, cols_tasks=cols_av)  # order?
    
    
    sample_no_NaNs = drop_na_in_cols(sample, default_covariates_t)
    sample_sz_no_NaNs = drop_na_in_cols(sample[sample[group_target_col] == 1], 
                                        default_covariates_r)

    columns_for_corr = cols_av + scale_cols
    # corrs
    corr_res = (sample[columns_for_corr]
                .corr(method='pearson')
                .sort_index(level=list(range(2)))
                .sort_index(axis=1, level=list(range(2))))
    # null (intercept-only) logistic log likelihood
    paired_llh_null = compute_logistic_llh_null(sample_no_NaNs, group_target_col)
    
    # best verbosity logistic r squared
    bvlr = compute_logistic_llh(sample_no_NaNs, group_target_col,
                                     [best_verbosity_metric] + default_covariates_t)
    bv_lr_adj = compute_ps_r_adj_llh(paired_llh_null, bvlr, k=len([best_verbosity_metric] + default_covariates_t))
    iteration_result[(f'llh_lr_bv', 'scale_independent', ('overview', 'col_independent'))] = bv_lr_adj
    # default covariates logistic r squared
    dclr = compute_logistic_llh(sample_no_NaNs, group_target_col, default_covariates_t)
    dc_lr_adj = compute_ps_r_adj_llh(paired_llh_null, dclr, k=len(default_covariates_t))
    iteration_result[(f'llh_lr_dc', 'scale_independent', ('overview', 'col_independent'))] = dc_lr_adj



    # Multivar
    # Logistic regression
    multivarlr = compute_logistic_llh(sample_no_NaNs, 
                                           group_target_col,
                                           [best_verbosity_metric] + 
                                           default_covariates_t + cols_to_test_additionally)
    multivar_ps_r_adj = compute_ps_r_adj_llh(paired_llh_null, multivarlr, k=len([best_verbosity_metric] + 
                                         default_covariates_t + cols_to_test_additionally))
    iteration_result[(f'llh_lr_multi', 'scale_independent', ('overview', 'col_independent'))] = multivar_ps_r_adj

    
    # Multivar
    # OLS
    multivarols, multivar_ols_adj = compute_OLS_rsquared(sample_sz_no_NaNs, target_col_corr,
                                       [best_verbosity_metric] + 
                                       default_covariates_r + cols_to_test_additionally)
    iteration_result[(f'r_ols_multi', target_col_corr, ('overview', 'col_independent'))] = multivar_ols_adj
    
    # OLS single best predcitor
    bvols, bv_ols_adj = compute_OLS_rsquared(sample_sz_no_NaNs, target_col_corr, [best_verbosity_metric] + default_covariates_r)
    iteration_result[(f'r_ols_bv', target_col_corr, ('overview', 'col_independent'))] = bv_ols_adj
    
    # OLS single default covariates
    dcols, dc_ols_adj = compute_OLS_rsquared(sample_sz_no_NaNs, target_col_corr, default_covariates_r)
    iteration_result[(f'r_ols_dc', target_col_corr, ('overview', 'col_independent'))] = dc_ols_adj

    for col in cols_av:
        #ttest
        iteration_result[('t', 'scale_independent', col)] = t_test_res[col]
        # auc
        iteration_result[(f'auc', 'scale_independent', col)] = compute_roc_auc(sample_no_NaNs, group_target_col, col)
        
        # Logistic regression incremental improvement
        if col != best_verbosity_metric:
            _, r_v_c, r_diff, r_marginal_diff, ps_r_adj = logistic_llh_difference(
                                                        sample_no_NaNs, group_target_col, 
                                                        [col], best_verbosity_metric, default_covariates_t, 
                                                        llh_control_only=bvlr,
                                                        llh_null=paired_llh_null)
            # (verbosity + metric) - (verbosity only)
            iteration_result[(f'llh_lr_diff', 'scale_independent', col)] = r_diff
            # (verbosity + metric)
            iteration_result[(f'llh_lr_bv_m', 'scale_independent', col)] = ps_r_adj  
            # (metric alone) - (intercept only)
            iteration_result[(f'llh_lr_marginal_diff', 'scale_independent', col)] = r_marginal_diff

        
        for col_to_correct_for in columns_to_correct_for:
            if col == col_to_correct_for:
                continue
            r_c = corr_res.loc[col, col_to_correct_for]
            iteration_result[(f'r_corr_w_{col_to_correct_for[-1]}', 'scale_independent', col)] = r_c

        for scale in scale_cols:
            iteration_result[('r', scale, col)] = corr_res.loc[col, scale]
            if scale != target_col_corr:
                continue
            if col == best_verbosity_metric:
                continue
            _, r_v_c, r_diff, r_marginal, r_adj = compute_OLS_rsquared_difference( 
                                                   sample_sz_no_NaNs, 
                                                   target_col_corr, 
                                                   [col], 
                                                   best_verbosity_metric, 
                                                   default_covariates_r, 
                                                   bvols)
            # (verbosity + metric) - (verbosity only)
            iteration_result[(f'r_ols_diff', target_col_corr, col)] = r_diff
            # (verbosity + metric)
            iteration_result[(f'r_ols_bv_m', target_col_corr, col)] = r_adj
            # (metric alone)                                                          
            iteration_result[(f'r_ols_marginal', target_col_corr, col)] = r_marginal 
    return iteration_result

In [43]:
def bootstrap_iteration(sample: pd.DataFrame,
                        group: str,
                        cols_av: list[tuple[str, str]],
                        scale_cols: list[tuple[str, str]],
                        columns_to_correct_for: list[tuple[str, str]],
                        best_verbosity_metric: tuple[str, str],
                        default_covariates_r: list[tuple[str, str]],
                        default_covariates_t: list[tuple[str, str]],
                        cols_to_test_additionally: list[tuple[str, str]],
                        target_col_corr: list[tuple[str, str]]) -> Optional[dict[tuple[str, str, str], float]]:
    warnings.filterwarnings(action="ignore")
    warnings.simplefilter('ignore')
    try:
        return try_bootstrap_iteration(sample, group, cols_av, scale_cols, columns_to_correct_for,
                                  best_verbosity_metric, default_covariates_r, default_covariates_t,
                                  cols_to_test_additionally, target_col_corr)
    except ValueError:
        return None

In [44]:
@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    """Context manager to patch joblib to report into tqdm progress bar given as argument"""

    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_batch_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_batch_callback
        tqdm_object.close()

In [45]:
def bootstrap_parallel(df: pd.DataFrame,
                       N: int,
                       group: str,
                       cols_av: list[tuple[str, str]],
                       scale_cols: list[tuple[str, str]],
                       columns_to_correct_for: list[tuple[str, str]],
                       best_verbosity_metric: tuple[str, str],
                       default_covariates_r: list[tuple[str, str]],
                       default_covariates_t: list[tuple[str, str]],
                       cols_to_test_additionally: list[tuple[str, str]],
                       target_col_corr: list[tuple[str, str]]) -> list[dict[tuple[str, str, str], float]]:
    results = []
    seed = 0
    while len(results) < N:
        total = N - len(results)
        presampled = [draw_sample_with_replacement(df, seed=seed + i, group_col=('target', group)) for i in
                      range(total)]
        with tqdm_joblib(tqdm(desc='progress', total=total)) as progress_bar:
            results_w_none = Parallel(n_jobs=-1)(
                delayed(bootstrap_iteration)(sample, group, cols_av, scale_cols, columns_to_correct_for,
                                      best_verbosity_metric, default_covariates_r, default_covariates_t,
                                      cols_to_test_additionally, target_col_corr)
                for sample in presampled)
        results.extend(filter(lambda x: x is not None, results_w_none))
        seed += N
    return results

In [46]:
scale_cols_ = [('target', x) for x in scale_cols]

In [47]:
# # an example for testing
# N = 10
# ex_res = apply_to_all_tasks(combined_data_all, bootstrap_parallel, N=N, cols_av=cols_av, scale_cols=scale_cols_, columns_to_correct_for=cols_to_correct_for, group='group',
#                            best_verbosity_metric=best_verbosity_metric, default_covariates_r=default_covariates_r,
#                            default_covariates_t=default_covariates_t, cols_to_test_additionally=cols_to_test_additionally,
#                            target_col_corr=('target', 'panss_total'))

**expensive to compute**

In [48]:
# # new procedure
# N = 1000
# dict_scales_unprocessed_samples = apply_to_all_tasks(combined_data_all, bootstrap_parallel, N=N, cols_av=cols_av, scale_cols=scale_cols_, columns_to_correct_for=cols_to_correct_for, group='group',
#                            best_verbosity_metric=best_verbosity_metric, default_covariates_r=default_covariates_r,
#                            default_covariates_t=default_covariates_t, cols_to_test_additionally=cols_to_test_additionally,
#                            target_col_corr=('target', 'panss_total'))

In [49]:
# # new procedure
# measures = set(measure for measure, scale, metric in dict_scales_unprocessed_samples['anger'][0].keys())
# reform = {key: {metric: [] for metric in list(cols_av) + [('overview', 'col_independent')]} for key in
#           [(task, scale[1], measure) for task in TASKS for scale in scale_cols_ for measure in measures]}
# for task, task_data_ in dict_scales_unprocessed_samples.items():
#     for item in task_data_:  # 1000 examples, we need to create lists of length 1000 instead
#         for measure, scale, metric in item.keys():
#             value = item[(measure, scale, metric)]
#             if scale == 'scale_independent':  # put scale_independent into each scale
#                 for _, scale_ in scale_cols_:
#                     reform[(task, scale_, measure)][metric].append(value)
#             else:
#                 reform[(task, scale[1], measure)][metric].append(value)

In [50]:
# # new procedure
# with open(f'processed_values/de_task_scales_samples_1000_w_verbosity_separate_reform_w_dep_w_multi_r_adj.pickle', 'wb') as f:
#     pickle.dump(reform, f)

In [51]:
# old procedure
# N = 1000
# dict_scales_unprocessed_samples = apply_to_all_tasks(combined_data_all, bootstrap_parallel,
#                                                      N=N, cols_av=cols_av, scale_cols=scale_cols_, columns_to_correct_for=cols_to_correct_for, group='group')
#
# with open(f'processed_values/de_task_scales_samples_{N}_w_verbosity_separate__.pickle', 'wb') as f:
#     pickle.dump(dict_scales_unprocessed_samples, f)

In [52]:
# old procedure
# measures = set(measure for measure, scale, metric in dict_scales_unprocessed_samples['anger'][0].keys())
# reform = {key: {metric: [] for metric in cols_av} for key in
#           [(task, scale[1], measure) for task in TASKS for scale in scale_cols_ for measure in measures]}
# for task, task_data_ in dict_scales_unprocessed_samples.items():
#     for item in task_data_:  # 1000 examples, we need to create lists of length 1000 instead
#         for measure, scale, metric in item.keys():
#             value = item[(measure, scale, metric)]
#             if scale == 'scale_independent':  # put scale_independent into each scale
#                 for _, scale_ in scale_cols_:
#                     reform[(task, scale_, measure)][metric].append(value)
#             else:
#                 reform[(task, scale[1], measure)][metric].append(value)

In [53]:
# old procedure
# with open(f'processed_values/de_task_scales_samples_1000_w_verbosity_separate_reform.pickle', 'wb') as f:
#     pickle.dump(reform, f)

In [54]:
# # old procedure
# with open(f'processed_values/de_task_scales_samples_1000_w_verbosity_separate_reform.pickle', 'rb') as f:
#     reform_ = pickle.load(f)

In [55]:
# # new procedure
with open(f'processed_values/de_task_scales_samples_1000_w_verbosity_separate_reform_w_dep_w_multi_r_adj.pickle', 'rb') as f:
    reform_ = pickle.load(f)

In [56]:
with open('processed_values/de_scales_samples_wo_o_tasks_w_verbosity.pickle', 'rb') as f:
    reform_tasks_v = pickle.load(f)

In [57]:
reformed_tasks_v = pd.DataFrame(reform_tasks_v)
reformed_tasks_v.columns.names = ['TASK', 'scale', 'measure']

In [58]:
reformed_tasks = pd.DataFrame(reform_) ## new sampling for tasks
reformed_tasks.columns.names = ['TASK', 'scale', 'measure']

# Plot & Analyze

In [59]:
figprms = {'syntactic': 
               {'subplot_size': (9, 4.5),
                'wspace': 0.25,
                'hspace': 0.125,
                'yt': 0.925
                }, 
           'LM': 
               {'subplot_size': (9, 5.5),
                'wspace': 0.275,
                'hspace': 0.125,
                'yt': 0.92
               }, 
           'lexical': 
               {'subplot_size': (9, 2),
                'wspace': 0.2,
                'hspace': 0.25,
                'yt': 0.925
               }, 
           'graph': 
               {'subplot_size': (9, 3.5), 
                'wspace': 0.3,
                'hspace': 0.125,
                'yt': 0.925
               }}

In [60]:
def get_fparams(m_type, n_sublots_height, n_sublots_width, figparams):
    subplot_size = figprms[m_type]['subplot_size']
    width = subplot_size[0] * n_sublots_width
    height = subplot_size[1] * n_sublots_height
    figsize = (width, height)
    wspace = figprms[m_type]['wspace']
    hspace = figprms[m_type]['hspace']
    yt = figprms[m_type]['yt']
    return figsize, wspace, hspace, yt

### Plot horizontal bar plots

In [61]:
def plot_horizontal_tasks(df, title, measure, m_type='syntactic', plot_abs=False, figparams=figprms,
                         errorbar=ERRORBAR):
    figsize, wspace, hspace, yt = get_fparams(m_type, 3, 2, figparams)
    fig, axes = plt.subplots(3, 2, figsize=figsize, sharex=True)
    fig.suptitle(title, y=yt+0.25)
    plt.subplots_adjust(wspace=wspace) #left=None, bottom=None, right=None, top=None, wspace=None, hspace=None)
    
    ab = 'abs ' if plot_abs else ''
    
    axs = axes.flatten()

    for i, scale in enumerate(ORDERED_SCALES):
        ax = axs[i]
        d = prep_horizontal_pointplot_errobar_data(df[scale].loc[m_type], measure, plot_abs=plot_abs)
        pointplot_horizontal(d, x=measure, ax=ax, errorbar=errorbar)
        ax.set_title(f'{ab}{measure} {scale}')
    
    add_grey(axes)

    if plot_abs:
        for ax in axes.reshape(-1): 
            ax.set_xlabel('abs ' + measure);
    return fig

In [62]:
verbosity_control_cols = ['r_corr_w_mean_sent_len', 'r_corr_w_n_sents', 'r_corr_w_n_words']

In [63]:
control_col_names = ['mean sentence length', 'sentence count', 'word count']

In [64]:
def plot_all(df, m_type='syntactic', measure='r', path=PATH_FIG, dpi=150, plot_abs=False, figparams=figprms,
             control_cols=['r_corr_w_control'], control_col_names=['mean sentence length'],
             errorbar=ERRORBAR):
    ab = 'abs_' if plot_abs else ''
    absolute_value = f' (absolute {measure} value)' if plot_abs else ''
    if len(control_cols) != len(control_col_names):
        raise ValueError('The names of the columns must match the columns in length.')
        
    fig = plot_horizontal_tasks(df, title=f'cross-scale comparison for {m_type} metrics{absolute_value}', 
                                measure=measure, m_type=m_type, plot_abs=plot_abs, figparams=figprms, 
                                errorbar=errorbar)
    plt.savefig(f'{path}{m_type}/{ab}scale_r.png', dpi=dpi)
    plt.close(fig)
    
    figsize, wspace, hspace, yt = get_fparams(m_type, 1, 1, figparams)
    d_t = prep_horizontal_pointplot_errobar_data(df['panss_o'].loc[m_type], 't')
    d_t['t'] = d_t['t'] * -1
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    fig.suptitle(f'group difference (t-test) for {m_type} metrics')
    pointplot_horizontal(d_t, 't', ax=ax, errorbar=errorbar)
    add_grey(ax, r=2)
    plt.savefig(f'{path}{m_type}/t.png', dpi=dpi, bbox_inches = 'tight')
    plt.close(fig)
    
    figsize, wspace, hspace, yt = get_fparams(m_type, len(control_cols), 1, figparams)
    fig, axes = plt.subplots(len(control_cols), figsize=figsize)
    fig.suptitle(f'correlation with verbosity for {m_type} metrics', y=yt-0.02)
    plt.subplots_adjust(wspace=wspace, hspace=hspace+0.08)
    for i, control_col in enumerate(control_cols):
        ax = axes[i] if len(control_cols) > 1 else axes
        name = control_col_names[i]
        d_c = prep_horizontal_pointplot_errobar_data(df['panss_o'].loc[m_type], control_col)

        pointplot_horizontal(d_c, control_col, ax=ax, errorbar=errorbar)
        ax.set_title(f'correlation with {name}');
        ax.set_xlabel('r');
        add_grey(ax)
    plt.savefig(f'{path}{m_type}/corr_verbosity.png', dpi=dpi, bbox_inches = 'tight')
    plt.close(fig)

    figsize, wspace, hspace, yt = get_fparams(m_type, len(control_cols), 2, figparams)
    fig, axes = plt.subplots(len(control_cols), 2, figsize=figsize)
    fig.suptitle(f'group difference and correlation with verbosity for {m_type} metrics', y=yt+0.02)
    plt.subplots_adjust(wspace=wspace, hspace=hspace+0.2)
    for i, control_col in enumerate(control_cols):
        ax = axes[i, 0] if len(control_cols) > 1 else axes[0]
        name = control_col_names[i]
        d_c = prep_horizontal_pointplot_errobar_data(df['panss_o'].loc[m_type], control_col)

        pointplot_horizontal(d_c, control_col, ax=ax, errorbar=errorbar)
        ax.set_title(f'correlation with {name}');
        ax.set_xlabel('r');
        add_grey(ax)

        ax_2 = axes[i, 1] if len(control_cols) > 1 else axes[1]
        pointplot_horizontal(d_t, x='t', ax=ax_2, errorbar=errorbar)
        ax_2.set_title('group difference (t-test)')
        add_grey(ax_2, r=2)
    plt.savefig(f'{path}{m_type}/t_test_corr_verbosity.png', dpi=dpi, bbox_inches = 'tight')
    plt.close(fig)

In [65]:
for m_type in reformed_d.index.unique(level=0):
    plot_all(reformed_d_w_verbosity, m_type, plot_abs=True,
                                 control_cols=verbosity_control_cols, control_col_names=control_col_names,
                                 errorbar=ERRORBAR)
    plot_all(reformed_d_w_verbosity, m_type, plot_abs=False,
                                 control_cols=verbosity_control_cols, control_col_names=control_col_names,
                                 errorbar=ERRORBAR)

In [66]:
for m_type in reformed_d.index.unique(level=0):
    plot_all(reformed_d, m_type, plot_abs=True, errorbar=ERRORBAR)
    plot_all(reformed_d, m_type, plot_abs=False, errorbar=ERRORBAR)

### Plot vertical bar plots for LMs

In [67]:
order = ['bert', 'glove_tf', 'glove_avg', 'w2v_tf', 'w2v_avg']

In [68]:
def plot_LM_scales(df, title, measure='r', plot_abs=False, figsize=(18, 18), order=order, errorbar=ERRORBAR):
    ab = 'abs ' if plot_abs else ''
    absolute_value = f' (absolute {measure} value)' if plot_abs else ''
    
    fig, axes = plt.subplots(3, 2, figsize=figsize, sharey=True)
    fig.suptitle(title + absolute_value, y=0.91)
    plt.subplots_adjust(wspace=0.075)
    
    axs = axes.flatten()
    for i, scale in enumerate(ORDERED_SCALES):
        ax = axs[i]
        d = prep_LM_pointplot(df.loc['LM', scale], measure, plot_abs=plot_abs)
        pointplot(d, x='model', y=measure, hue='metric', ax=ax, order=order, use_errorbar=True, errorbar=errorbar)
        ax.set_title(f'{ab}{measure} {scale}')
    
    add_grey(axes, line_dir='h')
    if plot_abs:
        for ax in axes.reshape(-1): 
            ax.set_ylabel('abs ' + measure);
    return fig

In [69]:
def plot_all_LM(df, path=PATH_FIG, dpi=150, plot_abs=False, figsize=(9, 9), measure='r',
                         control_cols=['r_corr_w_control'], control_col_names=['mean sentence length'],
               figparams=figprms, errorbar=ERRORBAR):
    
    ab = 'abs_' if plot_abs else ''
    if len(control_cols) != len(control_col_names):
        raise ValueError('The names of the columns must match the columns in length.')
    
    fig = plot_LM_scales(df, 'cross-scale comparison of LM metrcis across models', plot_abs=plot_abs, errorbar=errorbar)
    plt.savefig(f'{path}LM/model/{ab}scale_r.png', dpi=dpi, bbox_inches = 'tight')
    plt.close(fig)
    
    d_lm_t = prep_LM_pointplot(df.loc['LM', 'panss_o'], 't')
    d_lm_t['t'] = d_lm_t['t'] * -1
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    fig.suptitle('group difference (t-test) for LM metrcis across models')
    pointplot(d_lm_t, x='model', y='t', hue='metric', ax=ax, order=order, use_errorbar=True, errorbar=errorbar)
    add_grey(ax, r=2, line_dir='h')
    plt.savefig(f'{path}LM/model/t.png', dpi=dpi, bbox_inches = 'tight')
    plt.close(fig)

    absolute_value = f' (absolute {measure} value)' if plot_abs else ''
    figsize, wspace, hspace, yt = get_fparams(m_type, len(control_cols), 1, figparams)
    fig, axes = plt.subplots(len(control_cols), 1, figsize=(9, 15))
    fig.suptitle('correlation with verbosity for LM metrcis across models' + absolute_value, y=yt-0.01)
    plt.subplots_adjust(hspace=hspace+0.1)
    for i, control_col in enumerate(control_cols):
        name = control_col_names[i]
        ax = axes[i] if len(control_cols) > 1 else axes
        d_lm_c = prep_LM_pointplot(df.loc['LM', 'panss_o'], control_col, plot_abs=plot_abs)
        pointplot(d_lm_c, x='model', y=control_col, hue='metric', ax=ax, order=order, use_errorbar=True, errorbar=errorbar)
        ax.set_title(f'correlation with {name}')
        add_grey(ax, line_dir='h');
    plt.savefig(f'{path}LM/model/{ab}corr_verbosity.png', dpi=dpi, bbox_inches = 'tight')
    plt.close(fig)
    

    figsize, wspace, hspace, yt = get_fparams(m_type, len(control_cols), 2, figparams)
    fig, axes = plt.subplots(len(control_cols), 2, figsize=(18, 15))
    fig.suptitle('correlation with verbosity for LM metrcis across models' + absolute_value, y=yt-0.01)
    plt.subplots_adjust(wspace=wspace-0.18, hspace=hspace+0.08)
    for i, control_col in enumerate(control_cols):
        name = control_col_names[i]
        ax = axes[i, 0] if len(control_cols) > 1 else axes[0]
        d_lm_c = prep_LM_pointplot(df.loc['LM', 'panss_o'], control_col, plot_abs=plot_abs)
        pointplot(d_lm_c, x='model', y=control_col, hue='metric', ax=ax, order=order, use_errorbar=True, errorbar=errorbar)
        ax.set_title(f'correlation with {name}')
        add_grey(ax, line_dir='h');

        ax_2 = axes[i, 1] if len(control_cols) > 1 else axes[1]
        pointplot(d_lm_t, x='model', y='t', hue='metric', ax=ax_2, order=order, use_errorbar=True, errorbar=errorbar)
        ax_2.set_title('group difference (t-test)')
        add_grey(ax_2, r=2, line_dir='h')
    plt.savefig(f'{path}LM/model/t_test_corr_verbosity.png', dpi=dpi, bbox_inches = 'tight')
    plt.close(fig)

In [70]:
plot_all_LM(reformed_d_w_verbosity, plot_abs=True, 
            control_cols=verbosity_control_cols, control_col_names=control_col_names, errorbar=ERRORBAR)
plot_all_LM(reformed_d_w_verbosity, plot_abs=False,
            control_cols=verbosity_control_cols, control_col_names=control_col_names, errorbar=ERRORBAR)

In [71]:
plot_all_LM(reformed_d, plot_abs=True, errorbar=ERRORBAR)
plot_all_LM(reformed_d, plot_abs=False, errorbar=ERRORBAR)

## Cross-metric comparison

### R squared

In [72]:
def select_ok_metrics(row, t=1, r=0.3, rc=0.3):
    ok_corr = False
    ok_t = abs(row['saps_total', 't']) >= t
    ok_len = pd.isna(row['saps_total', 'r_corr_w_control']) or abs(row['saps_total', 'r_corr_w_control']) <= rc
    for scale in scale_cols:
        if abs(row[scale, 'r']) >= r:
            ok_corr = True 
            break
    return (ok_corr or ok_t) and ok_len

In [73]:
def select_ok_metrics_for_one_scale(row, scale, r=0.3, rc=0.3):
    ok_corr = False
    ok_len = pd.isna(row['panss_total', 'r_corr_w_control']) or abs(row['panss_total', 'r_corr_w_control']) <= rc
    ok_corr = abs(row[scale, 'r']) >= r
    return ok_corr and ok_len

In [74]:
def select_control_corr_ms_better_than_len(row, row_len, r=0.3, rc=0.3):
    if select_ok_metrics(row, r=r, rc=rc):
        return False
    ok_len = pd.isna(row['saps_total', 'r_corr_w_control']) or abs(row['saps_total', 'r_corr_w_control']) <= rc
    if ok_len:
        return False
    else:
        for scale in scales_:
            if abs(row[scale, 'r']) >= abs(row_len[scale, 'r']):
                return True
        return False

In [75]:
def select_control_corr_ms_better_than_len_for_one_scale(row, row_len, scale, r=0.3, rc=0.3):
    if select_ok_metrics(row, r=r, rc=rc):
        return False
    ok_len = pd.isna(row['saps_total', 'r_corr_w_control']) or abs(row['saps_total', 'r_corr_w_control']) <= rc
    if ok_len:
        return False
    else:
        if abs(row[scale, 'r']) >= abs(row_len[scale, 'r']) and abs(row[scale, 'r']) > r:
            return True
        return False

In [76]:
def select_bad_len_metrics(row, rc=0.3):
    ok_len = pd.isna(row['panss_total', 'r_corr_w_control']) or abs(row['panss_total', 'r_corr_w_control']) <= rc
    return not ok_len

In [77]:
median_d = reformed_d.applymap(np.nanmedian)

/var/folders/1l/2khv4cgj7xs0zrm9hnl_yrzh0000gn/T/ipykernel_1105/1276607103.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  median_d = reformed_d.applymap(np.nanmedian)


In [78]:
idxs_scale = {}
idxs_bad_bet_len = {}
for scale in scale_cols:
    ids = median_d[median_d.apply(lambda x: select_ok_metrics_for_one_scale(x, scale=scale), axis=1)].index
    idxs_scale[scale] = ids
    idsl = median_d[median_d.apply(lambda x: select_control_corr_ms_better_than_len_for_one_scale(x, 
                                                                                                  row_len=median_d.loc[('syntactic', 'mean_sent_len')], 
                                                                                                  scale=scale), axis=1)].index
    idxs_bad_bet_len[scale] = idsl

In [79]:
good_ms = sorted(list(set([y for x in idxs_scale.values() for y in x])))

In [80]:
bad_ms_better_than_len = sorted(list(set([y for x in idxs_bad_bet_len.values() for y in x])))

In [81]:
bad_ms = median_d[median_d.apply(select_bad_len_metrics, axis=1)].index

In [82]:
fig, ax = plt.subplots(1, 1, figsize=(8, 6), sharex=True)
fig.suptitle('cross-type metric comparison for correlation with mean sent len')
measure = 'r_corr_w_control'
plot_abs = False


d = prep_horizontal_pointplot_errobar_data(reformed_d['panss_pos'].loc[bad_ms], measure, plot_abs=plot_abs)
pointplot_horizontal(d, x=measure, ax=ax, errorbar=ERRORBAR)
ax.set_xlabel('r')
add_grey(ax)
plt.savefig(f'{PATH_FIG}/compare_corr_len.png', dpi=150, bbox_inches = 'tight')
plt.close(fig)

In [83]:
ms_to_plot = sorted(bad_ms_better_than_len + good_ms)

In [84]:
len(ms_to_plot)

21

In [85]:
len(good_ms)

5

In [86]:
{s: len(idxs_scale[s]) + len(idxs_bad_bet_len[s]) for s in idxs_scale}

{'saps_total': 4,
 'sans_total': 10,
 'panss_pos': 1,
 'panss_neg': 14,
 'panss_o': 14,
 'panss_total': 17}

In [87]:
{s: len(idxs_scale[s]) for s in idxs_scale}

{'saps_total': 1,
 'sans_total': 3,
 'panss_pos': 0,
 'panss_neg': 4,
 'panss_o': 2,
 'panss_total': 3}

In [88]:
def sort_index(idxs):
    ms = sorted(idxs)
    for s in ('syntactic: mean_sent_len', 'mean_sent_len', ('syntactic', 'mean_sent_len')):
        if s in ms:
            ms.remove(s)
            ms.append(s)
    return ms 

In [89]:
def map_marker(m, scale, idxs_scale):
    if m in idxs_scale[scale]:
        return 'o'
    else: 
        return 'x'

In [90]:
def add_len_lines(ax, median_d, scale, measure='r'):
    ax.axvline(median_d.loc[('syntactic', 'mean_sent_len')][scale, measure], linestyle='--')
    ax.axvline(-median_d.loc[('syntactic', 'mean_sent_len')][scale, measure], linestyle='--')

In [91]:
def plot_one_scale(ax, scale, idxs_scale, reformed_d, ms_to_plot, measure, plot_abs, use_markers=True, errorbar=ERRORBAR):
    markers = [map_marker(m, scale, idxs_scale) for m in ms_to_plot]
    if ('syntactic', 'mean_sent_len') in idxs_scale[scale]:
        add_len_lines(ax, median_d, scale)
    d = prep_horizontal_pointplot_errobar_data(reformed_d[scale].loc[ms_to_plot], measure, plot_abs=plot_abs)
    pointplot_horizontal(d, x=measure, ax=ax, markers=markers if use_markers else 'o', errorbar=errorbar)
    ax.set_title(f'{measure} {scale}')

In [92]:
fig, axes = plt.subplots(3, 2, figsize=(18, 14), sharex=True)
fig.suptitle('cross-scale cross-type metric comparison', y=0.915)
plt.subplots_adjust(wspace=0.4, hspace=0.1) #left=None, bottom=None, right=None, top=None, wspace=None, hspace=None)
measure = 'r'
plot_abs = False
idxs = sort_index(ms_to_plot)
axs = axes.flatten()

for i, scale in enumerate(ORDERED_SCALES):
    ax = axs[i]
    plot_one_scale(ax, scale, idxs_scale, reformed_d, ms_to_plot, measure, plot_abs, errorbar=ERRORBAR)
    
add_grey(axes)
plt.savefig(f'{PATH_FIG}/compare_r.png', dpi=150, bbox_inches = 'tight')
plt.close(fig)

#### Images for the Presentation

In [93]:
idx_to_plot = [m for m in ms_to_plot if m not in [('syntactic', 'AUX'),  ('LM', 'w2v_tf_cgcoh')]]

In [94]:
fig, axes = plt.subplots(1, 2, figsize=(18, 4))
plt.subplots_adjust(wspace=0.4)
fig.suptitle('cross-type metric comparison for correlation with SANS and inverse mean sentence length', y=1)

dcr = reformed_d['sans_total'].loc[idx_to_plot].drop([('syntactic', 'mean_sent_len')])
d_c = prep_horizontal_pointplot_errobar_data(dcr, 'r_corr_w_control', plot_abs=False)
d_c['r_corr_w_control'] = -1 * d_c['r_corr_w_control']

plot_one_scale(axes[0], 'sans_total', idxs_scale, reformed_d, idx_to_plot, measure, plot_abs, use_markers=False)

pointplot_horizontal(d_c, x='r_corr_w_control', ax=axes[1], palette='tab20', errorbar=ERRORBAR)
axes[1].set_title('-1 * r mean sentence length')
axes[1].set_xlabel('-r');
add_grey(axes)
plt.savefig(f'{PATH_FIG}/compare_sans_to_minus_corr_len_more_metrics.png', dpi=150, bbox_inches = 'tight')
plt.close(fig)

In [95]:
idxs_sans = sorted([m for m in idxs_scale['sans_total']] + [m for m in idxs_bad_bet_len['sans_total']])

In [96]:
fig, axes = plt.subplots(1, 2, figsize=(18, 4))
plt.subplots_adjust(wspace=0.4)
fig.suptitle('cross-type metric comparison for correlation with SANS and inverse mean sentence length', y=1)

dcr = reformed_d['sans_total'].loc[idxs_sans].drop([('syntactic', 'mean_sent_len')])
d_c = prep_horizontal_pointplot_errobar_data(dcr, 'r_corr_w_control', plot_abs=False)
d_c['r_corr_w_control'] = -1 * d_c['r_corr_w_control']

plot_one_scale(axes[0], 'sans_total', idxs_scale, reformed_d, idxs_sans, measure, plot_abs, use_markers=False)

pointplot_horizontal(d_c, x='r_corr_w_control', ax=axes[1], palette='tab20', errorbar=ERRORBAR)
axes[1].set_title('-1 * r mean sentence length')
axes[1].set_xlabel('-r');
add_grey(axes)
plt.savefig(f'{PATH_FIG}/compare_sans_to_minus_corr_len.png', dpi=150, bbox_inches = 'tight')
plt.close(fig)

### t-test

In [97]:
def select_ok_metrics_t(row, errrorbar=('pi', 50)):
    width = errrorbar[1]
    crop = (100-width) / 2
    q_low_n = crop/100
    q_high_n = (100-crop) / 100
    q_low = np.quantile(row['saps_total', 't'], q_low_n)
    q_high = np.quantile(row['saps_total', 't'], q_high_n)
    return q_low > 0 or q_high < 0

In [98]:
idx_comp_t = reformed_d[reformed_d.apply(select_ok_metrics_t, axis=1)].index

In [99]:
## 50 PI used
fig, axes = plt.subplots(1, 2, figsize=(18, 4))
plt.subplots_adjust(wspace=0.4)
fig.suptitle('cross-type metric comparison for group difference and correlation with mean sentence length', y=1)

dcr = reformed_d['sans_total'].loc[idx_comp_t].drop([('syntactic', 'mean_sent_len')])
d_c = prep_horizontal_pointplot_errobar_data(dcr, 'r_corr_w_control', plot_abs=False)
d_t = prep_horizontal_pointplot_errobar_data(reformed_d['sans_total'].loc[idx_comp_t], 't', plot_abs=False)
d_t['t'] = d_t['t'] * -1

pointplot_horizontal(d_t, x='t', ax=axes[0], errorbar=ERRORBAR)
axes[0].set_title('t')
axes[0].set_title('group difference (t-test)')
add_len_lines(axes[0], median_d, scale, 't')

pointplot_horizontal(d_c, x='r_corr_w_control', ax=axes[1], errorbar=ERRORBAR)
axes[1].set_title('correlation with mean sentence length')
axes[1].set_xlabel('r');
add_grey(axes[0], r=2)
add_grey(axes[1])
plt.savefig(f'{PATH_FIG}/compare_t.png', dpi=150, bbox_inches = 'tight')
plt.close(fig)

#### Images for the Presentation

In [100]:
fig, axes = plt.subplots(1, 2, figsize=(18, 4))
plt.subplots_adjust(wspace=0.4)
fig.suptitle('cross-type metric comparison for group difference and correlation with inverse mean sentence length', y=1)

dcr = reformed_d['sans_total'].loc[idx_comp_t].drop([('syntactic', 'mean_sent_len')])
d_c = prep_horizontal_pointplot_errobar_data(dcr, 'r_corr_w_control', plot_abs=False)
d_t = prep_horizontal_pointplot_errobar_data(reformed_d['sans_total'].loc[idx_comp_t], 't', plot_abs=False)
d_t['t'] = d_t['t'] * -1
d_c['r_corr_w_control'] = -1 * d_c['r_corr_w_control']

pointplot_horizontal(d_t, x='t', ax=axes[0], errorbar=ERRORBAR)
axes[0].set_title('t')
axes[0].set_title('t-test')
add_len_lines(axes[0], median_d, scale, 't')

pointplot_horizontal(d_c, x='r_corr_w_control', ax=axes[1], errorbar=ERRORBAR)
axes[1].set_title('-1 * r mean sentence length')
axes[1].set_xlabel('-r');
add_grey(axes[0], r=2)
add_grey(axes[1])
plt.savefig(f'{PATH_FIG}/compare_t_minus_corr_len.png', dpi=150, bbox_inches = 'tight')
plt.close(fig)

### average LM model / metric performance medians across scales

In [101]:
scales_ = ('sans_total', 'saps_total', 'panss_pos', 'panss_neg', 'panss_o', 'panss_total')
sc_ind_ = ('t', 'r_corr_w_control')
models_ = ('bert', 'glove_tf', 'glove_avg', 'w2v_tf', 'w2v_avg')
metrics_ = ('cgcoh', 'gcoh', 'lcoh', 'scoh', 'sprob', 'pppl')

In [102]:
def mean_model_metric_medians(median_df, leave_out=()):
    resp_d_model = pd.DataFrame(columns=[f'{sc} abs r' for sc in scales_] + list(sc_ind_), index=models_)
    resp_d_metric = pd.DataFrame(columns=[f'{sc} abs r' for sc in scales_] + list(sc_ind_), index=metrics_)
    for scale in scales_:
        ex_d = prep_LM_pointplot(median_df.loc['LM', scale], col='r', use_errorbar=False, plot_abs=True)
        for model in models_:
            leave_out_ = ex_d[ex_d['model'] == model]
            leave_out_ = leave_out_[~leave_out_['metric'].isin(leave_out)]
            resp_d_model.loc[model, f'{scale} abs r'] = np.nanmean(leave_out_['r'])
        for metric in metrics_:
            resp_d_metric.loc[metric, f'{scale} abs r'] = np.nanmean(ex_d[ex_d['metric'] == metric]['r'])
    for sc_ind in sc_ind_:
        for model in models_:
            leave_out_ = ex_d[ex_d['model'] == model]
            leave_out_ = leave_out_[~leave_out_['metric'].isin(leave_out)]
            resp_d_model.loc[model, sc_ind] = np.nanmean(leave_out_[sc_ind])
        for metric in metrics_:
            resp_d_metric.loc[metric, sc_ind] = np.nanmean(ex_d[ex_d['metric'] == metric][sc_ind])
    return resp_d_model, resp_d_metric


#### only including cosine-similarity based metrics

In [103]:
resp_d_model, resp_d_metric = mean_model_metric_medians(median_d, leave_out=('pppl', 'sprob'))

/Users/galina.ryazanskaya/Downloads/thesis?/code?/jupyter_utils.py:289: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(abs)
/Users/galina.ryazanskaya/Downloads/thesis?/code?/jupyter_utils.py:289: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(abs)
/Users/galina.ryazanskaya/Downloads/thesis?/code?/jupyter_utils.py:289: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(abs)
/Users/galina.ryazanskaya/Downloads/thesis?/code?/jupyter_utils.py:289: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(abs)
/Users/galina.ryazanskaya/Downloads/thesis?/code?/jupyter_utils.py:289: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(abs)
/Users/galina.ryazanskaya/Downloads/thesis?/code?/jupyter_utils.py:289: FutureWarning: DataFrame.applymap has 

In [104]:
resp_d_model

,sans_total abs r,saps_total abs r,panss_pos abs r,panss_neg abs r,panss_o abs r,panss_total abs r,t,r_corr_w_control
bert,0.236577,0.04418,0.07079,0.225726,0.252314,0.226234,0.511409,0.157991
glove_tf,0.165904,0.208092,0.234331,0.192519,0.164074,0.198569,0.39171,0.429151
glove_avg,0.224203,0.195832,0.200804,0.234329,0.184796,0.23216,0.394616,0.567291
w2v_tf,0.280315,0.303707,0.258881,0.312226,0.254946,0.324928,0.729381,0.592533
w2v_avg,0.306171,0.246063,0.189779,0.314721,0.233608,0.291453,0.613459,0.618035


In [105]:
resp_d_model[['sans_total abs r',
 'saps_total abs r',
 'panss_pos abs r',
 'panss_neg abs r',
 'panss_o abs r',
 'panss_total abs r']].mean(axis=1).sort_values() 

bert          0.17597
glove_tf     0.193915
glove_avg    0.212021
w2v_avg      0.263633
w2v_tf       0.289167
dtype: object

In [106]:
resp_d_model['r_corr_w_control']

bert         0.157991
glove_tf     0.429151
glove_avg    0.567291
w2v_tf       0.592533
w2v_avg      0.618035
Name: r_corr_w_control, dtype: object

In [107]:
resp_d_metric['r_corr_w_control']

cgcoh    0.292816
gcoh     0.493656
lcoh     0.549157
scoh     0.556372
sprob    0.576114
pppl     0.529623
Name: r_corr_w_control, dtype: object

#### including feature based metrics for BERT

In [108]:
resp_d_model, resp_d_metric = mean_model_metric_medians(median_d)

/Users/galina.ryazanskaya/Downloads/thesis?/code?/jupyter_utils.py:289: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(abs)
/Users/galina.ryazanskaya/Downloads/thesis?/code?/jupyter_utils.py:289: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(abs)
/Users/galina.ryazanskaya/Downloads/thesis?/code?/jupyter_utils.py:289: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(abs)
/Users/galina.ryazanskaya/Downloads/thesis?/code?/jupyter_utils.py:289: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(abs)
/Users/galina.ryazanskaya/Downloads/thesis?/code?/jupyter_utils.py:289: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(abs)
/Users/galina.ryazanskaya/Downloads/thesis?/code?/jupyter_utils.py:289: FutureWarning: DataFrame.applymap has 

In [109]:
resp_d_metric

,sans_total abs r,saps_total abs r,panss_pos abs r,panss_neg abs r,panss_o abs r,panss_total abs r,t,r_corr_w_control
cgcoh,0.105246,0.190676,0.142618,0.101842,0.085081,0.098206,0.872924,0.292816
gcoh,0.176989,0.182976,0.163138,0.200634,0.159281,0.204929,0.134382,0.493656
lcoh,0.314356,0.232184,0.256773,0.328579,0.292895,0.344763,0.729779,0.549157
scoh,0.373945,0.192464,0.201138,0.392562,0.334534,0.370777,0.375375,0.556372
sprob,0.245612,0.131085,0.202997,0.318265,0.278489,0.314706,1.28745,0.576114
pppl,0.471239,0.155275,0.119219,0.454768,0.432778,0.412121,2.038195,0.529623


In [110]:
resp_d_model[['sans_total abs r',
 'saps_total abs r',
 'panss_pos abs r',
 'panss_neg abs r',
 'panss_o abs r',
 'panss_total abs r']].mean(axis=1).sort_values() 

glove_tf     0.193915
glove_avg    0.212021
bert         0.215551
w2v_avg      0.263633
w2v_tf       0.289167
dtype: object

In [111]:
resp_d_model['r_corr_w_control']

bert         0.289617
glove_tf     0.429151
glove_avg    0.567291
w2v_tf       0.592533
w2v_avg      0.618035
Name: r_corr_w_control, dtype: object

In [112]:
resp_d_metric['r_corr_w_control']

cgcoh    0.292816
gcoh     0.493656
lcoh     0.549157
scoh     0.556372
sprob    0.576114
pppl     0.529623
Name: r_corr_w_control, dtype: object

## Plot and analyze across parts of NET

In [113]:
verbosity_control_cols

['r_corr_w_mean_sent_len', 'r_corr_w_n_sents', 'r_corr_w_n_words']

In [114]:
def plot_horizontal_tasks(df, title, scale, measure, xname=None, m_type='syntactic', 
                          plot_abs=False, r=0.3, figparams=figprms, 
                          control_cols = ['r_corr_w_control'], control_col_names=['mean sentence length'],
                          errorbar=ERRORBAR):
    absolute_value = f' (absolute r value)' if plot_abs else ''
    if len(control_cols) != len(control_col_names):
        raise ValueError('The names of the columns must match the columns in length.')
    
    figsize, wspace, hspace, yt = get_fparams(m_type, 2, 2, figparams)
    fig, axes = plt.subplots(2, 2, figsize=figsize, sharex=True)
    fig.suptitle(title + absolute_value, y=yt)
    plt.subplots_adjust(wspace=wspace, hspace=hspace)
    
    axs = axes.flatten()
    
    for i, task in enumerate(TASKS):
        ax = axs[i]
        data_task = df.loc[m_type, (task, scale)]
        if m_type == 'syntactic':
            if measure == 'r_corr_w_mean_sent_len' or measure == 'r_corr_w_control':
                data_task = df.loc[m_type, (task, scale)].drop('mean_sent_len')
            elif measure == 'r_corr_w_mean_sent_len':
                data_task = df.loc[m_type, (task, scale)].drop('n_sents')
        elif m_type == 'lexical' and measure == 'r_corr_w_n_words':
            data_task = df.loc[m_type, (task, scale)].drop('n_words')
            
        data = prep_horizontal_pointplot_errobar_data(data_task, col=measure, plot_abs=plot_abs)
        if measure == 't':
            data['t'] = data['t'] * -1
        pointplot_horizontal(data, x=measure, ax=ax, errorbar=errorbar)
        ax.set_title(task)

    add_grey(axes, r=r)
    if xname is None:
        xname = measure
    
    for ax in axes.reshape(-1): 
        label = 'abs ' + xname if plot_abs else xname
        ax.set_xlabel(label)
    return fig

In [115]:
m_type = 'syntactic'
fig = plot_horizontal_tasks(reformed_tasks, 
                      f'cross-task comparison for {m_type} metrics on group difference (t-test)', 
                      scale='panss_o', measure='t', m_type=m_type, r=2, figparams=figprms,
                      errorbar=ERRORBAR)
plt.close(fig)

In [116]:
fig = plot_horizontal_tasks(reformed_tasks, 
                      f'cross-task comparison for {m_type} metrics on sans_total', 
                      scale='sans_total', measure='r', m_type=m_type, figparams=figprms,
                      errorbar=ERRORBAR)
plt.close(fig)

In [117]:
fig = plot_horizontal_tasks(reformed_tasks, 
                            f'cross-task comparison for {m_type} metrics on correlation with mean sentence length', 
                            scale='panss_o', measure='r_corr_w_mean_sent_len',
                            xname='r', m_type=m_type, figparams=figprms,
                            errorbar=ERRORBAR)
plt.close(fig)

In [118]:
def plot_all_scales(reformed_d, m_type='syntactic', path=PATH_FIG, plot_abs=True, dpi=150, figparams=figprms,
                    control_cols=['r_corr_w_control'], control_col_names=['mean sentence length'],
                    errorbar=ERRORBAR):
    
    fig = plot_horizontal_tasks(reformed_d, 
                                f'cross-task comparison for {m_type} metrics on group difference (t-test)', 
                                scale='panss_o', measure='t', m_type=m_type, r=2, figparams=figparams, errorbar=errorbar)
    plt.savefig(f'{path}{m_type}/t_across_tasks.png', dpi=dpi, bbox_inches = 'tight')
    plt.close(fig)
    
    ab = 'abs_' if plot_abs else ''
    
    upfig = figparams.copy()
    if m_type == 'lexical':
        upfig[m_type]['yt'] += 0.05
    for i, control_col in enumerate(control_cols):
        name = control_col_names[i]
        fig = plot_horizontal_tasks(reformed_d, 
                                f'cross-task comparison for {m_type} metrics on correlation with {name}', 
                                scale='panss_o', measure=control_col, plot_abs=plot_abs, 
                                xname='r', m_type=m_type, figparams=upfig, errorbar=errorbar)
        plt.savefig(f'{path}{m_type}/{ab}corr_{"_".join(name.split())}_across_tasks.png', 
                    dpi=dpi, bbox_inches = 'tight')
        plt.close(fig)
    
    for scale in ORDERED_SCALES:
        fig = plot_horizontal_tasks(reformed_d, 
                                    f'cross-task comparison for {m_type} metrics on {scale}', 
                                    scale=scale, measure='r', 
                                    plot_abs=plot_abs, m_type=m_type, figparams=figparams, errorbar=errorbar)
        plt.savefig(f'{path}{m_type}/{ab}r_{scale}_across_tasks.png', dpi=dpi, bbox_inches = 'tight')
        plt.close(fig)

In [119]:
for m_type in reformed_tasks.index.unique(level=0):
    if m_type == 'overview':
        continue
    plot_all_scales(reformed_tasks, m_type, plot_abs=True, control_cols=verbosity_control_cols, control_col_names=control_col_names, errorbar=ERRORBAR)
    plot_all_scales(reformed_tasks, m_type, plot_abs=False, control_cols=verbosity_control_cols, control_col_names=control_col_names, errorbar=ERRORBAR)

In [120]:
def plot_lm_tasks(df, title, scale, measure, order=order, plot_abs=False, use_errorbar=True,
                 figsize=(15, 10), yname=None, r=0.3, errorbar=ERRORBAR):
    absolute_value = f' (absolute {measure} value)' if plot_abs else ''
    fig, axes = plt.subplots(2, 2, figsize=figsize, sharey=True)
    fig.suptitle(title+absolute_value, y=0.925)
    plt.subplots_adjust(wspace=0.1)
    
    axs = axes.flatten()
    for i, task in enumerate(TASKS):
        ax = axs[i]
        d = prep_LM_pointplot(df.loc['LM', (task, scale)], col=measure, plot_abs=plot_abs)
        if measure == 't':
            d['t'] = d['t'] * -1
        pointplot(d, x='model', y=measure, hue='metric', ax=ax, order=order, use_errorbar=use_errorbar, errorbar=errorbar)
        ax.set_title(task)

    add_grey(axes, line_dir='h', r=r)
    
    if yname is None:
        yname = measure
    for ax in axes.reshape(-1): 
        label = 'abs ' + yname if plot_abs else yname
        ax.set_ylabel(label)
    return fig

In [121]:
fig = plot_lm_tasks(reformed_tasks, 
                    'cross-task comparison for LM metrics across models on group difference (t-test)',
                    scale='panss_o', measure='t', use_errorbar=True, figsize=(15, 10), r=2, errorbar=ERRORBAR)
plt.close(fig)

In [122]:
fig = plot_lm_tasks(reformed_tasks, 
                    'cross-task comparison for LM metrics across models on sans_total',
                    scale='sans_total', measure='r', use_errorbar=True, figsize=(15, 10), errorbar=ERRORBAR)
plt.close(fig)

In [123]:
def plot_all_LM_across_tasks(reformed_d, m_type='LM', path=PATH_FIG, plot_abs=False, figsize=(18, 12), dpi = 150,
                    control_cols=['r_corr_w_control'], control_col_names=['mean sentence length'], errorbar=ERRORBAR):
    ab = 'abs_' if plot_abs else ''
    absolute_value = f' (absolute r value)' if plot_abs else ''
    
    fig = plot_lm_tasks(reformed_d, 
                        'cross-task comparison for LM metrics across models on group difference (t-test)',
                        scale='panss_o', measure='t', use_errorbar=True, figsize=figsize, r=2, errorbar=errorbar)
    plt.savefig(f'{path}{m_type}/model/t_across_tasks.png', dpi=dpi, bbox_inches = 'tight')
    plt.close(fig)
    
    for i, control_col in enumerate(control_cols):
        name = control_col_names[i]
        fig = plot_lm_tasks(reformed_d, 
                            f'cross-task comparison for LM metrics across models on correlation with {name}',
                            scale='panss_o', measure=control_col, yname='r',
                            plot_abs=plot_abs, use_errorbar=True, figsize=figsize, errorbar=errorbar)
        plt.savefig(f'{path}{m_type}/model/{ab}corr_{"_".join(name.split())}_across_tasks.png', 
                    dpi=dpi, bbox_inches = 'tight')
        plt.close(fig)
    
    for scale in ORDERED_SCALES:
        fig = plot_lm_tasks(reformed_d, 
                            f'cross-task comparison for LM metrics across models on {scale}{absolute_value}', 
                            scale=scale, measure='r', 
                            plot_abs=plot_abs, use_errorbar=True, figsize=figsize, errorbar=errorbar)
        plt.savefig(f'{path}{m_type}/model/{ab}r_{scale}_across_tasks.png', dpi=dpi, bbox_inches = 'tight')
        plt.close(fig)
    plt.close(fig)

In [124]:
plot_all_LM_across_tasks(reformed_tasks, plot_abs=True, control_cols=verbosity_control_cols, control_col_names=control_col_names, errorbar=ERRORBAR)
plot_all_LM_across_tasks(reformed_tasks, plot_abs=False, control_cols=verbosity_control_cols, control_col_names=control_col_names, errorbar=ERRORBAR)

# Reform the dataset to long format

index: unique

langugae: de / ru

task: (de tasks) / (ru tasks) - 4 for each

scale: (de: panss sans saps t test) / (ru: panss dep td t test)

metric: name

matric_group: 4

values: median CI_low CI_high corr_mean_sent_len corr_n_sent corr_n_word

In [125]:
control_corr_names = ['r_corr_w_mean_sent_len', 'r_corr_w_n_sents', 'r_corr_w_n_words']

In [126]:
low = 0.025
high = 0.975
lang = 'de'

In [127]:
TASKS

['anger', 'fear', 'happiness', 'sadness']

In [128]:
ORDERED_SCALES

['panss_pos',
 'panss_neg',
 'panss_o',
 'panss_total',
 'saps_total',
 'sans_total']

In [129]:
reg_metrics_grouped = ['r']

reg_metrics_add_to_all = ['r_corr_w_mean_sent_len', 'r_corr_w_n_sents', 'r_corr_w_n_words']

reg_metrics_ols_m = ['r_ols_bv_m', 'r_ols_diff', 'r_ols_marginal'] # write that it is sz only in the long data

In [130]:
col_independent_metrics = ['r_ols_multi', 'r_ols_bv', 'r_ols_dc', 
                           'llh_lr_multi', 'llh_lr_bv', 'llh_lr_dc',
#                            'llh_ps_r_bv', 
#                            'llh_ps_r_multi'
                          ]

In [131]:
group_diff_metrics = ['t', 'auc', 'llh_lr_bv_m', 'llh_lr_diff', 'llh_lr_marginal_diff', 
#                       'llh_ps_r_bv_m'
                     ]

In [132]:
# reform_[('happiness', 'panss_total', 'r')][('syntactic', 'min_sent_len')]

In [133]:
def process_data(reform_, data, control_corr_names):
    median = np.nanmedian(data)
    mean = np.nanmean(data)
    CI_low = np.nanquantile(np.array(data), low)
    CI_high = np.nanquantile(np.array(data), high)
#   if np.isnan(median):
#       print('nan median in: ', task, scale_, performance_metric, metric_group, metric_name)
#   line = (lang, task, scale_, metric_name)
    control_cols_medians, control_cols_means, control_cols_CI_lows, control_cols_CI_highs = {}, {}, {}, {}
    for control_col in control_corr_names:
        control_data = reform_[(task, scale_key, control_col)][metric]
        if not control_data:
            c_median, c_mean, c_CI_high, c_CI_low = np.nan, np.nan, np.nan, np.nan
        else:
            c_median = np.nanmedian(control_data)
            c_mean = np.nanmean(control_data)
            c_CI_low = np.nanquantile(np.array(control_data), low)
            c_CI_high = np.nanquantile(np.array(control_data), high)
        control_cols_medians[control_col] = c_median
        control_cols_means[control_col] = c_mean
        control_cols_CI_lows[control_col] = c_CI_low
        control_cols_CI_highs[control_col] = c_CI_high
    return median, mean, CI_low, CI_high, control_cols_medians, control_cols_means, control_cols_CI_lows, control_cols_CI_highs

In [134]:
def form_long_line(reform_, metric, task, scale_, scale_key, performance_metric_, performance_metric_key, predictor_varset, control_corr_names):
    metric_group, metric_name = metric
    data = reform_[(task, scale_key, performance_metric_key)][(metric_group, metric_name)]
    median, mean, CI_low, CI_high, control_cols_medians, control_cols_means, control_cols_CI_lows, control_cols_CI_highs = process_data(reform_, data, control_corr_names)
    long_line = (lang, task, scale_, performance_metric_, predictor_varset, metric_group, metric_name, 
                 median, mean, CI_low, CI_high)
    for control_col in control_corr_names:
        long_line += (control_cols_medians[control_col], control_cols_means[control_col], 
                      control_cols_CI_lows[control_col], control_cols_CI_highs[control_col])
    return long_line

In [135]:
def update_long_data(long_data, reform_, metric, task, scale_, scale_key, performance_metric_, performance_metric_key, predictor_varset, control_corr_names):
    long_line = form_long_line(reform_, metric, task, scale_, scale_key, performance_metric_, performance_metric_key, predictor_varset, control_corr_names)
    if long_line not in long_data:
        long_data.append(long_line)
    else:
        print(long_line)

In [136]:
pm_varset_map = {
    "r": ("r", "single"),
    
    "t": ("t", "single"),
    
    "auc": ("auc", "single"),
    
    "llh_lr_bv_m": ("llh", "dc_bv_m"),
    
    "llh_lr_diff": ("llh_diff", "single"),
    "llh_lr_marginal_diff": ("llh_marginal_diff", "single"),
    
    "llh_lr_multi": ("llh", "multi"),
    "llh_lr_bv": ("llh", "dc_bv"),
    "llh_lr_dc": ("llh", "dc"),
    
    "r_ols_bv_m": ("r2", "dc_bv_m"),
    
    "r_ols_diff": ("r2_diff", "single"),
    
    "r_ols_marginal": ("r_marginal", "single"),
    
    "r_ols_multi": ("r2", "multi"),
    "r_ols_bv": ("r2", "dc_bv"),
    "r_ols_dc": ("r2", "dc"),
    
}

In [137]:
long_data = []
for task in TASKS:
    # ___regressions___
    # r
    for scale_ in ORDERED_SCALES:
        performance_metric_key = "r"
        scale_key = scale_
        performance_metric_, predictor_varset = pm_varset_map[performance_metric_key]
        for metric in cols_av:
            update_long_data(long_data, reform_, metric, task, scale_, scale_key, performance_metric_, performance_metric_key, predictor_varset, control_corr_names)

    # OLS
    # r squared, r squared diff
    # ['r_ols_bv_m', 'r_ols_diff', 'r_ols_marginal']
    for performance_metric_key in reg_metrics_ols_m:
        scale_key = 'panss_total'  # because target_col_corr=('target', 'panss_total')
        scale_ = 'panss_total'
        performance_metric_, predictor_varset = pm_varset_map[performance_metric_key]
        for metric in cols_av:
            update_long_data(long_data, reform_, metric, task, scale_, scale_key, performance_metric_, performance_metric_key, predictor_varset, control_corr_names)

    # ___group_diff___
    # t, auc, llh logistic, llh logistic diff, llh logistic marginal diff
    for pm in group_diff_metrics:
        scale_ = 'group_diff'           
        scale_key = 'panss_total'
        performance_metric_key = pm
        performance_metric_, predictor_varset = pm_varset_map[performance_metric_key]
        for metric in cols_av:
            update_long_data(long_data, reform_, metric, task, scale_, scale_key, performance_metric_, performance_metric_key, predictor_varset, control_corr_names)

    # ___col independent___
    # OLS r squared multi
    # OLS best verbosity
    # OLS default covariates
    # llh logistic multi
    # llh best verbosity
    # llh default covariates
    metric = ('overview', 'col_independent')
    scale_key = 'panss_total'
    scale_ = 'multivariate'
    for performance_metric_key in col_independent_metrics:
        performance_metric_, predictor_varset = pm_varset_map[performance_metric_key]
        update_long_data(long_data, reform_, metric, task, scale_, scale_key, performance_metric_, performance_metric_key, predictor_varset, control_corr_names)

In [138]:
long_df = pd.DataFrame(long_data, columns=('lang', 'task', 'scale', 'performance_metric', 
                                           'predictor_varset', 'metric_group', 'metric_name', 
                                           'median', 'mean', 'CI_low', 'CI_high',
                                           'corr_mean_sent_len_median', 'corr_mean_sent_len_mean', 
                                           'corr_mean_sent_len_CI_low', 'corr_mean_sent_len_CI_high',
                                           'corr_n_sents_median', 'corr_n_sents_mean',
                                           'corr_n_sents_CI_low', 'corr_n_sents_CI_high',
                                           'corr_n_words_median', 'corr_n_words_mean',
                                           'corr_n_words_CI_low', 'corr_n_words_CI_high',))
long_df.tail()

,lang,task,scale,performance_metric,predictor_varset,metric_group,metric_name,median,mean,CI_low,...,corr_mean_sent_len_CI_low,corr_mean_sent_len_CI_high,corr_n_sents_median,corr_n_sents_mean,corr_n_sents_CI_low,corr_n_sents_CI_high,corr_n_words_median,corr_n_words_mean,corr_n_words_CI_low,corr_n_words_CI_high
2875,de,sadness,multivariate,r2,dc_bv,overview,col_independent,0.226627,0.241840,0.017281,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2876,de,sadness,multivariate,r2,dc,overview,col_independent,0.162010,0.185736,-0.005605,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2877,de,sadness,multivariate,llh,multi,overview,col_independent,0.537601,0.542434,0.261121,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2878,de,sadness,multivariate,llh,dc_bv,overview,col_independent,0.409773,0.409516,0.158642,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2879,de,sadness,multivariate,llh,dc,overview,col_independent,0.081603,0.101622,-0.024363,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [139]:
long_df[long_df[['lang', 'task', 'scale', 'performance_metric', 'predictor_varset', 'metric_group', 'metric_name']].duplicated(keep=False)]

,lang,task,scale,performance_metric,predictor_varset,metric_group,metric_name,median,mean,CI_low,...,corr_mean_sent_len_CI_low,corr_mean_sent_len_CI_high,corr_n_sents_median,corr_n_sents_mean,corr_n_sents_CI_low,corr_n_sents_CI_high,corr_n_words_median,corr_n_words_mean,corr_n_words_CI_low,corr_n_words_CI_high


In [140]:
long_df.scale.unique()

array(['panss_pos', 'panss_neg', 'panss_o', 'panss_total', 'saps_total',
       'sans_total', 'group_diff', 'multivariate'], dtype=object)

In [141]:
long_df.performance_metric.unique()

array(['r', 'r2', 'r2_diff', 'r_marginal', 't', 'auc', 'llh', 'llh_diff',
       'llh_marginal_diff'], dtype=object)

In [142]:
long_df.predictor_varset.unique()

array(['single', 'dc_bv_m', 'multi', 'dc_bv', 'dc'], dtype=object)

In [143]:
long_df.to_csv(PATH + '/long_de_95_multi_rekeyed_r_adj.csv')